In [ ]:
# takeing selected classes form the folder with changed labeling
import os
import shutil
import yaml
from pathlib import Path

# -------------------------------------------------------
# INPUT CHILLI DATASET
# -------------------------------------------------------
dataset = r"F:\0Defense\Dataset\Vegetables_Dataset_seperate_folder\Chilli"

# -------------------------------------------------------
# OUTPUT FILTERED FOLDER
# -------------------------------------------------------
merged_dir = r"F:\0Defense\Dataset\Merged\Chilli_Filtered"

# Create output folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{merged_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{merged_dir}/{split}/labels", exist_ok=True)

# -------------------------------------------------------
# TARGET CLASSES YOU WANT
# -------------------------------------------------------
TARGET_CLASSES = [
    "Curling-Leafs",
    "Healthy-Leafs",
    "Leaf Spot-Leafs",
    "Leveillula taurica-Leafs",
    "Mozaik Leaf"
]

# -------------------------------------------------------
# Load class list from YAML
# -------------------------------------------------------
def load_yaml_classes(path):
    with open(path, "r") as f:
        data = yaml.safe_load(f)
    return data["names"]

# Mapping for index remap
# Original → New
index_remap = {2:0, 3:1, 4:2, 5:3, 6:4}

# -------------------------------------------------------
# Function to process train/valid/test
# -------------------------------------------------------
def process_split(split_name, src_root, dst_root, target_classes):

    src_labels = Path(src_root) / split_name / "labels"
    src_images = Path(src_root) / split_name / "images"

    dst_labels = Path(dst_root) / split_name / "labels"
    dst_images = Path(dst_root) / split_name / "images"

    yaml_file = Path(src_root) / "data.yaml"
    class_names = load_yaml_classes(yaml_file)

    # class_name → class_id
    class_to_id = {name: i for i, name in enumerate(class_names)}

    # Find desired IDs from target class names
    desired_ids = []
    for name in target_classes:
        if name in class_to_id:
            desired_ids.append(class_to_id[name])
        else:
            print(f"[WARNING] Class not found in YAML: {name}")

    print(f"\n[INFO] Processing {split_name} → desired IDs: {desired_ids}")

    # Loop over all label files
    for label_file in src_labels.glob("*.txt"):

        with open(label_file, "r") as f:
            lines = f.readlines()

        if not lines:
            continue

        found = False
        new_label_lines = []

        # Process all lines in label file
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(parts[0])

            # If class is in our target list → remap it
            if cls_id in desired_ids:
                found = True
                new_cls = index_remap[cls_id]  # remap index
                parts[0] = str(new_cls)
                new_label_lines.append(" ".join(parts))

        # If label contains our desired classes
        if found:

            # Image lookup
            img_jpg = src_images / (label_file.stem + ".jpg")
            img_png = src_images / (label_file.stem + ".png")

            img_path = img_jpg if img_jpg.exists() else img_png if img_png.exists() else None

            if img_path is None:
                print(f"[WARNING] No image for {label_file.stem}")
                continue

            # Save modified label file
            with open(dst_labels / label_file.name, "w") as f:
                f.write("\n".join(new_label_lines))

            # Copy image
            shutil.copy(img_path, dst_images / img_path.name)


# -------------------------------------------------------
# RUN FOR train, valid, test
# -------------------------------------------------------
for split in ["train", "valid", "test"]:
    process_split(split, dataset, merged_dir, TARGET_CLASSES)

# -------------------------------------------------------
# CREATE FINAL YAML
# -------------------------------------------------------
final_yaml = {
    "train": "../train/images",
    "val": "../valid/images",
    "test": "../test/images",
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES
}

with open(f"{merged_dir}/data.yaml", "w") as f:
    yaml.dump(final_yaml, f, default_flow_style=False)

print("\n✅ DONE! Filtered dataset created successfully!")


In [ ]:
# marge 2 dataset in one folder
import os
import shutil
import yaml
from pathlib import Path

# -------------------------------------------------------
# DATASET PATHS
# -------------------------------------------------------
DATASET_1 = r"F:\0Defense\Dataset\Vegetables_Dataset_seperate_folder_increase_size\Tomato"
DATASET_2 = r"F:\0Defense\Dataset\3.1\Tomato_old"

# -------------------------------------------------------
# OUTPUT MERGED FOLDER
# -------------------------------------------------------
MERGED_DIR = r"F:\0Defense\Dataset\Merged\Tomato_Merged"

# Create output directories
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

# -------------------------------------------------------
# TARGET CLASSES FROM BOTH DATASETS
# FINAL ORDER
# -------------------------------------------------------
FINAL_CLASSES = [
    'bacterial_spot', 'curl_virus', 'Healthy'
]

# Map FINAL CLASSES to new indices
FINAL_CLASS_TO_INDEX = {name: i for i, name in enumerate(FINAL_CLASSES)}


# -------------------------------------------------------
# LOAD YAML
# -------------------------------------------------------
def load_yaml_classes(path):
    with open(path, "r") as f:
        data = yaml.safe_load(f)
    return data["names"]


# -------------------------------------------------------
# FILTER FUNCTION
# -------------------------------------------------------
def process_dataset(src_root, dst_root, target_class_names):

    yaml_file = Path(src_root) / "data.yaml"
    class_names = load_yaml_classes(yaml_file)

    # Map class name → original ID
    class_to_id = {name: i for i, name in enumerate(class_names)}

    # Gather original IDs for needed classes
    desired_ids = {
        class_to_id[name]: FINAL_CLASS_TO_INDEX[name]
        for name in target_class_names if name in class_to_id
    }

    print(f"\n📌 Processing {src_root}")
    print(f"✔ Desired class-ID mapping: {desired_ids}")

    for split in ["train", "valid", "test"]:
        src_labels = Path(src_root) / split / "labels"
        src_images = Path(src_root) / split / "images"

        dst_labels = Path(dst_root) / split / "labels"
        dst_images = Path(dst_root) / split / "images"

        if not src_labels.exists():
            continue

        for txt_file in src_labels.glob("*.txt"):

            with open(txt_file, "r") as f:
                lines = f.readlines()
        
            new_lines = []
            found = False
        
            for line in lines:
                parts = line.strip().split()
                old_id = int(parts[0])
        
                if old_id in desired_ids:
                    found = True
                    new_id = desired_ids[old_id]
                    parts[0] = str(new_id)
                    new_lines.append(" ".join(parts))
        
            if not found:
                continue
        
            # Find corresponding image
            img_jpg = src_images / (txt_file.stem + ".jpg")
            img_png = src_images / (txt_file.stem + ".png")
            img_path = None
        
            if img_jpg.exists(): img_path = img_jpg
            elif img_png.exists(): img_path = img_png
        
            if img_path is None:
                print(f"[WARNING] No image found for {txt_file.stem}")
                continue
        
            # Save label + image
            with open(dst_labels / txt_file.name, "w") as f:
                f.write("\n".join(new_lines))
        
            shutil.copy(img_path, dst_images / img_path.name)



# -------------------------------------------------------
# RUN PROCESSING
# -------------------------------------------------------

# Dataset 1 classes
DATASET_1_CLASSES = [
    'bacterial_spot', 'curl_virus'
]

# Dataset 2 class
DATASET_2_CLASSES = ['Healthy']

process_dataset(DATASET_1, MERGED_DIR, DATASET_1_CLASSES)
process_dataset(DATASET_2, MERGED_DIR, DATASET_2_CLASSES)


# -------------------------------------------------------
# WRITE FINAL YAML
# -------------------------------------------------------
final_yaml = {
    "train": "../train/images",
    "val": "../valid/images",
    "test": "../test/images",
    "nc": len(FINAL_CLASSES),
    "names": FINAL_CLASSES
}

with open(f"{MERGED_DIR}/data.yaml", "w") as f:
    yaml.dump(final_yaml, f)

print("\n✅ COMPLETED! Eggplant merged dataset created successfully!")
